# LeetCode 1299 — Replace Elements with Greatest Element on Right Side bằng Python

**Link đề:** https://leetcode.com/problems/replace-elements-with-greatest-element-on-right-side/
**Độ khó:** Easy · **Chủ đề:** Array

```python
class Solution:
    def replaceElements(self, arr: list[int]) -> list[int]:
        ...
```

## Mục tiêu học
1. Viết lời giải **brute force** bằng slicing `arr[i+1:]` + `max`, và tự đo được vì sao nó `O(n²)`.
2. Nhận ra **công việc lặp lại**: max của phần bên phải được tính đi tính lại.
3. Học kỹ thuật **duyệt ngược + giữ max chạy (running max)** — nền tảng của "suffix maximum".
4. Làm **tại chỗ (in-place)** với `O(1)` bộ nhớ phụ, biết **thứ tự cập nhật** để không làm mất dữ liệu.
5. Thêm góc nhìn `itertools.accumulate`; trace, chứng minh invariant, test đối chiếu, lỗi hay gặp, biến thể và bài tập.

## 1. Đề bài và ví dụ

Cho mảng `arr`, thay **mỗi phần tử** bằng **phần tử lớn nhất trong các phần tử bên phải nó**, và thay phần tử cuối cùng bằng `-1`. Trả về mảng sau khi thay.

| `arr` | Output |
|---|---|
| `[17,18,5,4,6,1]` | `[18,6,6,6,1,-1]` |
| `[400]` | `[-1]` |

Giải thích ví dụ 1:
- index 0 → lớn nhất bên phải là `18` (index 1).
- index 1 → lớn nhất bên phải là `6` (index 4).
- index 2, 3 → `6`.
- index 4 → bên phải chỉ có `1`.
- index 5 → không còn gì bên phải → `-1`.

**Constraints:** `1 <= arr.length <= 10^4`, `1 <= arr[i] <= 10^5`.

Chú ý: "bên phải" là **chỉ các phần tử sau** `i`, **không** gồm chính `arr[i]`.

In [1]:
arr = [17, 18, 5, 4, 6, 1]
for i in range(len(arr)):
    right = arr[i + 1:]
    print(f"i={i}, arr[i]={arr[i]:>2}, bên phải={right}, max={max(right) if right else -1}")

i=0, arr[i]=17, bên phải=[18, 5, 4, 6, 1], max=18
i=1, arr[i]=18, bên phải=[5, 4, 6, 1], max=6
i=2, arr[i]= 5, bên phải=[4, 6, 1], max=6
i=3, arr[i]= 4, bên phải=[6, 1], max=6
i=4, arr[i]= 6, bên phải=[1], max=1
i=5, arr[i]= 1, bên phải=[], max=-1


## 2. Nền tảng Python cần biết

### 2.1. Slicing `arr[i+1:]` — phần bên phải

`arr[a:]` lấy từ chỉ số `a` tới hết, tạo **list mới**. Khi `a == len(arr)` kết quả là list **rỗng** `[]` (không lỗi).

In [2]:
arr = [17, 18, 5, 4, 6, 1]
print(arr[1:])     # [18, 5, 4, 6, 1]
print(arr[5:])     # [1]
print(arr[6:])     # []  <- vượt cuối vẫn không lỗi

[18, 5, 4, 6, 1]
[1]
[]


### 2.2. `max` và trường hợp rỗng

`max([])` báo **`ValueError`**. Hai cách xử lý: kiểm tra rỗng trước, hoặc dùng tham số `default=-1`.

In [3]:
print(max([18, 5, 4]))
try:
    max([])
except ValueError as e:
    print("ValueError:", e)
print(max([], default=-1))     # -1

18
ValueError: max() iterable argument is empty
-1


### 2.3. Duyệt ngược: `range(n - 1, -1, -1)`

`range(start, stop, step)` với `step = -1` đi lùi; `stop` **không** được lấy, nên muốn tới `0` phải đặt `stop = -1`.
Cách khác: `reversed(range(n))`.

In [4]:
n = 6
print(list(range(n - 1, -1, -1)))   # [5, 4, 3, 2, 1, 0]
print(list(reversed(range(n))))     # cùng kết quả
print(list(range(n - 1, 0, -1)))    # [5, 4, 3, 2, 1] <- thiếu 0 nếu đặt stop = 0!

[5, 4, 3, 2, 1, 0]
[5, 4, 3, 2, 1, 0]
[5, 4, 3, 2, 1]


### 2.4. Hoán đổi / gán đồng thời

`a, b = b, a` tính **vế phải trước** rồi mới gán. Nên `arr[i], mx = mx, max(mx, arr[i])` dùng `arr[i]` **cũ** để tính `max`, rồi mới ghi đè `arr[i]`. Đây là mẹo quan trọng cho lời giải in-place.

In [5]:
arr = [5, 9]
mx = -1
arr[0], mx = mx, max(mx, arr[0])    # vế phải: (-1, max(-1, 5)) = (-1, 5)
print(arr, mx)                      # [-1, 9] 5

[-1, 9] 5


## 3. Cách 1 — Brute force: với mỗi i, tính `max(arr[i+1:])`

In [6]:
def replace_brute(arr: list[int]) -> list[int]:
    n = len(arr)
    res = []
    for i in range(n):
        res.append(max(arr[i + 1:], default=-1))
    return res

print(replace_brute([17, 18, 5, 4, 6, 1]))   # [18, 6, 6, 6, 1, -1]
print(replace_brute([400]))                  # [-1]

[18, 6, 6, 6, 1, -1]
[-1]


**Độ phức tạp:** với mỗi `i`, slicing + `max` tốn `n - i - 1` bước → tổng `(n-1) + (n-2) + … + 0 = n(n-1)/2` → **`O(n²)`** thời gian; bộ nhớ `O(n)` cho list tạm mỗi lần slicing.

Với `n = 10^4` là khoảng `5·10^7` bước — chậm trong Python (có thể vẫn AC nhưng sát giới hạn). Đếm thử:

In [7]:
def count_steps_brute(n):
    return sum(n - i - 1 for i in range(n))

for n in [10, 100, 1000, 10**4]:
    print(f"n={n:>6}: {count_steps_brute(n):>12,} bước so sánh")

n=    10:           45 bước so sánh
n=   100:        4,950 bước so sánh
n=  1000:      499,500 bước so sánh
n= 10000:   49,995,000 bước so sánh


`f"{x:,}"` in số có dấu phẩy ngăn cách hàng nghìn cho dễ đọc.

## 4. Nút thắt

Nhìn kỹ ví dụ: `max(arr[2:])`, `max(arr[3:])`, `max(arr[4:])`… Mỗi đoạn sau là **đoạn trước bỏ đi 1 phần tử đầu**. Ta đang tính lại gần như cùng một việc `n` lần.

Đảo hướng suy nghĩ: đi **từ phải sang trái**, đoạn bên phải **thêm** 1 phần tử mỗi bước:

```text
max(bên phải của i) = max( arr[i+1], max(bên phải của i+1) )
```

Chỉ cần nhớ một số: **max của mọi thứ đã đi qua** (running max).

In [8]:
arr = [17, 18, 5, 4, 6, 1]
suffix_max = -1
for i in range(len(arr) - 1, -1, -1):
    print(f"i={i}: max bên phải = {suffix_max:>2}, rồi gộp arr[{i}]={arr[i]} -> ", end="")
    suffix_max = max(suffix_max, arr[i])
    print(suffix_max)

i=5: max bên phải = -1, rồi gộp arr[5]=1 -> 1
i=4: max bên phải =  1, rồi gộp arr[4]=6 -> 6
i=3: max bên phải =  6, rồi gộp arr[3]=4 -> 6
i=2: max bên phải =  6, rồi gộp arr[2]=5 -> 6
i=1: max bên phải =  6, rồi gộp arr[1]=18 -> 18
i=0: max bên phải = 18, rồi gộp arr[0]=17 -> 18


`print(..., end="")` không xuống dòng, để `print` tiếp theo in nối vào cùng dòng.

## 5. Cách 2 — Mảng suffix max riêng (dễ hiểu, `O(n)` bộ nhớ)

Tạo `res` dài `n`. Duyệt `i` từ cuối về đầu:
1. `res[i] = mx` (max của phần bên phải, **chưa** gồm `arr[i]`).
2. `mx = max(mx, arr[i])` (gộp `arr[i]` để chuẩn bị cho `i - 1`).

In [9]:
def replace_suffix_array(arr: list[int]) -> list[int]:
    n = len(arr)
    res = [0] * n
    mx = -1
    for i in range(n - 1, -1, -1):
        res[i] = mx
        mx = max(mx, arr[i])
    return res

print(replace_suffix_array([17, 18, 5, 4, 6, 1]))   # [18, 6, 6, 6, 1, -1]
print(replace_suffix_array([400]))                  # [-1]

[18, 6, 6, 6, 1, -1]
[-1]


Vì sao khởi tạo `mx = -1`? Với phần tử cuối, "bên phải" rỗng và đề yêu cầu `-1`. Vì mọi `arr[i] >= 1 > -1`, giá trị `-1` không bao giờ "thắng" trong `max` sau khi đã gặp phần tử thật — nên vừa đúng biên, vừa không ảnh hưởng kết quả khác.

## 6. Cách 3 — Tại chỗ (in-place), `O(1)` bộ nhớ phụ (lời giải chính)

Ghi thẳng vào `arr`. Rủi ro: ghi `arr[i] = mx` **trước** thì mất giá trị cũ của `arr[i]` cần cho `max`. Cách xử lý: lưu tạm, hoặc gán đồng thời.

In [10]:
def replace_in_place(arr: list[int]) -> list[int]:
    mx = -1
    for i in range(len(arr) - 1, -1, -1):
        old = arr[i]          # giữ giá trị cũ trước khi ghi đè
        arr[i] = mx
        mx = max(mx, old)
    return arr

a = [17, 18, 5, 4, 6, 1]
print(replace_in_place(a))   # [18, 6, 6, 6, 1, -1]
print(a)                     # a đã bị sửa (cùng một list)

[18, 6, 6, 6, 1, -1]
[18, 6, 6, 6, 1, -1]


### Phiên bản gán đồng thời (một dòng)

In [11]:
def replace_in_place_tuple(arr: list[int]) -> list[int]:
    mx = -1
    for i in range(len(arr) - 1, -1, -1):
        arr[i], mx = mx, max(mx, arr[i])
    return arr

print(replace_in_place_tuple([17, 18, 5, 4, 6, 1]))   # [18, 6, 6, 6, 1, -1]

[18, 6, 6, 6, 1, -1]


**In-place nghĩa là gì với người gọi?** Hàm sửa **chính list được truyền vào** (Python truyền tham chiếu tới object). Nếu người gọi còn cần mảng gốc thì phải truyền bản sao `arr[:]`. Trên LeetCode chỉ cần giá trị trả về, nên sửa tại chỗ là ổn.

## 7. Cách 4 — `itertools.accumulate` (góc nhìn "prefix max trên mảng đảo")

`accumulate(xs, max)` cho dãy max cộng dồn: `[x0, max(x0,x1), max(x0,x1,x2), …]`. Áp dụng trên mảng **đảo ngược** ta có suffix max, rồi dịch đi 1 vị trí và thêm `-1`.

In [12]:
from itertools import accumulate

print(list(accumulate([3, 1, 4, 1, 5], max)))      # [3, 3, 4, 4, 5]

def replace_accumulate(arr: list[int]) -> list[int]:
    rev = arr[::-1]                                  # [1, 6, 4, 5, 18, 17]
    pref = list(accumulate(rev, max))                # max của 1, 2, 3, ... phần tử cuối
    # phần tử i (từ phải) cần max của (i) phần tử đứng sau nó -> dịch phải 1, thêm -1 ở đầu
    shifted = [-1] + pref[:-1]
    return shifted[::-1]

print(replace_accumulate([17, 18, 5, 4, 6, 1]))      # [18, 6, 6, 6, 1, -1]
print(replace_accumulate([400]))                     # [-1]

[3, 3, 4, 4, 5]
[18, 6, 6, 6, 1, -1]
[-1]


Cách này gọn nhưng tạo nhiều list tạm (`O(n)` bộ nhớ) và khó đọc hơn. Hữu ích để nhận ra: **suffix max = prefix max của mảng đảo**.

## 8. Trace từng bước (Cách 3)

`arr = [17, 18, 5, 4, 6, 1]`, `mx = -1`:

| i | `arr[i]` cũ | ghi `arr[i] = mx` | `mx = max(mx, cũ)` | `arr` sau bước |
|---:|---:|---:|---:|---|
| 5 | 1 | −1 | 1 | `[17,18,5,4,6,-1]` |
| 4 | 6 | 1 | 6 | `[17,18,5,4,1,-1]` |
| 3 | 4 | 6 | 6 | `[17,18,5,6,1,-1]` |
| 2 | 5 | 6 | 6 | `[17,18,6,6,1,-1]` |
| 1 | 18 | 6 | 18 | `[17,6,6,6,1,-1]` |
| 0 | 17 | 18 | 18 | `[18,6,6,6,1,-1]` |

In [13]:
def trace_replace(arr):
    arr = arr[:]                       # làm trên bản sao để không phá dữ liệu gốc
    mx = -1
    print(f"bắt đầu: arr={arr}, mx={mx}")
    for i in range(len(arr) - 1, -1, -1):
        old = arr[i]
        arr[i] = mx
        new_mx = max(mx, old)
        print(f"i={i}: cũ={old:>2} | ghi arr[{i}]={mx:>2} | mx: {mx:>2} -> {new_mx:>2} | arr={arr}")
        mx = new_mx
    return arr

trace_replace([17, 18, 5, 4, 6, 1])

bắt đầu: arr=[17, 18, 5, 4, 6, 1], mx=-1
i=5: cũ= 1 | ghi arr[5]=-1 | mx: -1 ->  1 | arr=[17, 18, 5, 4, 6, -1]
i=4: cũ= 6 | ghi arr[4]= 1 | mx:  1 ->  6 | arr=[17, 18, 5, 4, 1, -1]
i=3: cũ= 4 | ghi arr[3]= 6 | mx:  6 ->  6 | arr=[17, 18, 5, 6, 1, -1]
i=2: cũ= 5 | ghi arr[2]= 6 | mx:  6 ->  6 | arr=[17, 18, 6, 6, 1, -1]
i=1: cũ=18 | ghi arr[1]= 6 | mx:  6 -> 18 | arr=[17, 6, 6, 6, 1, -1]
i=0: cũ=17 | ghi arr[0]=18 | mx: 18 -> 18 | arr=[18, 6, 6, 6, 1, -1]


[18, 6, 6, 6, 1, -1]

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def replaceElements(self, arr: list[int]) -> list[int]:
        mx = -1
        for i in range(len(arr) - 1, -1, -1):
            arr[i], mx = mx, max(mx, arr[i])
        return arr

print(Solution().replaceElements([17, 18, 5, 4, 6, 1]))   # [18, 6, 6, 6, 1, -1]
print(Solution().replaceElements([400]))                  # [-1]

[18, 6, 6, 6, 1, -1]
[-1]


## 10. Vì sao đúng? (invariant)

**Invariant:** ngay **trước** khi xử lý chỉ số `i` (duyệt từ phải sang), `mx = max(arr_gốc[i+1:])`, với quy ước max của đoạn rỗng là `-1`.

- **Khởi đầu** (`i = n-1`): đoạn `arr[n:]` rỗng, `mx = -1`. Đúng.
- **Duy trì:** ta ghi `arr[i] = mx` — đúng định nghĩa đề. Sau đó `mx = max(mx, arr_gốc[i]) = max(arr_gốc[i:])` — đúng invariant cho bước `i - 1`.
- **Dữ liệu gốc không bị mất:** tại bước `i` ta chỉ đọc `arr[i]` **trước** khi ghi đè nó, và mọi `arr[j]` với `j < i` chưa bị chạm tới. Các ô `j > i` đã bị ghi đè nhưng thông tin của chúng đã được "gói" vào `mx`.
- **Kết thúc** sau `i = 0`: mọi ô đã đúng.

## 11. Độ phức tạp và so sánh

Gọi `n = len(arr)`.

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| 1. Brute force `max(arr[i+1:])` | `O(n²)` | `O(n)` (slice tạm) | dễ nghĩ |
| 2. Mảng suffix riêng | `O(n)` | `O(n)` | không sửa input |
| 3. In-place running max | `O(n)` | `O(1)` | lời giải chính |
| 4. `accumulate` trên mảng đảo | `O(n)` | `O(n)` | gọn, góc nhìn prefix/suffix |

In [15]:
import time, random

random.seed(0)
big = [random.randint(1, 10**5) for _ in range(5000)]
for f in (replace_brute, replace_suffix_array, replace_in_place):
    t0 = time.perf_counter()
    f(big[:])
    print(f"{f.__name__:<22} n=5000: {time.perf_counter() - t0:.4f} giây")

replace_brute          n=5000: 0.1817 giây
replace_suffix_array   n=5000: 0.0003 giây
replace_in_place       n=5000: 0.0003 giây


(Thời gian cụ thể phụ thuộc máy; điều cần thấy là brute force chậm hơn **nhiều lần** dù `n` mới chỉ 5000.)

## 12. Bộ test

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.replaceElements([17, 18, 5, 4, 6, 1]), [18, 6, 6, 6, 1, -1])
check("đề 2", sol.replaceElements([400]), [-1])
check("2 phần tử", sol.replaceElements([1, 2]), [2, -1])
check("tăng dần", sol.replaceElements([1, 2, 3, 4]), [4, 4, 4, -1])
check("giảm dần", sol.replaceElements([4, 3, 2, 1]), [3, 2, 1, -1])
check("bằng nhau", sol.replaceElements([5, 5, 5]), [5, 5, -1])
check("max ở cuối", sol.replaceElements([1, 1, 1, 9]), [9, 9, 9, -1])

PASS đề 1: [18, 6, 6, 6, 1, -1]
PASS đề 2: [-1]
PASS 2 phần tử: [2, -1]
PASS tăng dần: [4, 4, 4, -1]
PASS giảm dần: [3, 2, 1, -1]
PASS bằng nhau: [5, 5, -1]
PASS max ở cuối: [9, 9, 9, -1]


In [17]:
random.seed(1299)
methods = [replace_brute, replace_suffix_array, replace_in_place, replace_in_place_tuple, replace_accumulate]
for _ in range(1000):
    arr = [random.randint(1, 20) for _ in range(random.randint(1, 30))]
    expected = replace_brute(arr)
    for f in methods:
        assert f(arr[:]) == expected, (f.__name__, arr)       # truyền bản sao vì có hàm sửa tại chỗ
    assert Solution().replaceElements(arr[:]) == expected
print("1000 mảng ngẫu nhiên: 5 cách + Solution đều khớp")

1000 mảng ngẫu nhiên: 5 cách + Solution đều khớp


## 13. Lỗi người mới hay gặp

### Lỗi 1: Ghi đè trước khi lấy giá trị cũ

In [18]:
def wrong_overwrite_first(arr):
    mx = -1
    for i in range(len(arr) - 1, -1, -1):
        arr[i] = mx
        mx = max(mx, arr[i])            # SAI: arr[i] lúc này đã là mx, không còn giá trị gốc
    return arr

print("sai :", wrong_overwrite_first([17, 18, 5, 4, 6, 1]))
print("đúng:", Solution().replaceElements([17, 18, 5, 4, 6, 1]))

sai : [-1, -1, -1, -1, -1, -1]
đúng: [18, 6, 6, 6, 1, -1]


Kết quả sai là toàn `-1`: `mx` không bao giờ được cập nhật bằng giá trị thật.

### Lỗi 2: Cập nhật `mx` **trước** khi ghi (gộp luôn chính `arr[i]`)

Đề yêu cầu max **bên phải**, không gồm `arr[i]`.

In [19]:
def wrong_include_self(arr):
    mx = -1
    for i in range(len(arr) - 1, -1, -1):
        mx = max(mx, arr[i])            # SAI: gộp chính nó trước
        arr[i] = mx
    return arr

print("sai :", wrong_include_self([17, 18, 5, 4, 6, 1]))
print("đúng:", Solution().replaceElements([17, 18, 5, 4, 6, 1]))

sai : [18, 18, 6, 6, 6, 1]
đúng: [18, 6, 6, 6, 1, -1]


### Lỗi 3: Duyệt xuôi từ trái sang phải với running max

Running max từ trái cho max **bên trái**, không phải bên phải.

In [20]:
def wrong_left_to_right(arr):
    res, mx = [], -1
    for x in arr:
        res.append(mx)
        mx = max(mx, x)
    return res

print("sai :", wrong_left_to_right([17, 18, 5, 4, 6, 1]), "(đây là max bên TRÁI)")
print("đúng:", Solution().replaceElements([17, 18, 5, 4, 6, 1]))

sai : [-1, 17, 18, 18, 18, 18] (đây là max bên TRÁI)
đúng: [18, 6, 6, 6, 1, -1]


### Lỗi 4: `range(n - 1, 0, -1)` bỏ sót index 0

In [21]:
def wrong_stop_zero(arr):
    mx = -1
    for i in range(len(arr) - 1, 0, -1):     # SAI: không chạy tới i = 0
        arr[i], mx = mx, max(mx, arr[i])
    return arr

print("sai :", wrong_stop_zero([17, 18, 5, 4, 6, 1]), "<- phần tử đầu vẫn là 17")

sai : [17, 6, 6, 6, 1, -1] <- phần tử đầu vẫn là 17


### Lỗi 5: `max(arr[i+1:])` không có `default` → lỗi ở phần tử cuối

In [22]:
def wrong_no_default(arr):
    return [max(arr[i + 1:]) for i in range(len(arr))]

try:
    wrong_no_default([17, 18, 5])
except ValueError as e:
    print("ValueError:", e)

ValueError: max() iterable argument is empty


## 14. Mẹo nhận dạng pattern và biến thể

**Dấu hiệu:**
- "Lớn nhất / nhỏ nhất **bên phải** (hoặc bên trái) của mỗi phần tử" → duyệt từ phía đó, giữ **running max/min** (suffix/prefix max).
- "Phần tử lớn hơn **gần nhất** bên phải" (next greater element) → khác! Cần **monotonic stack** (ngăn xếp đơn điệu).
- Cần nhiều truy vấn max trên đoạn bất kỳ → sparse table / segment tree (nâng cao).

**Bài liên quan:**
- LeetCode 121 *Best Time to Buy and Sell Stock* (running min từ trái).
- LeetCode 42 *Trapping Rain Water* (prefix max + suffix max).
- LeetCode 496 / 503 / 739 *Next Greater Element*, *Daily Temperatures* (monotonic stack).
- LeetCode 1475 *Final Prices With a Special Discount* (next smaller-or-equal).

In [23]:
# Mini so sánh: "max bên phải" (bài này) vs "phần tử lớn hơn GẦN NHẤT bên phải" (next greater, LeetCode 496/503)
def next_greater(arr):
    res = [-1] * len(arr)
    stack = []                                  # chứa chỉ số, giá trị giảm dần từ đáy lên đỉnh
    for i, x in enumerate(arr):
        while stack and arr[stack[-1]] < x:
            res[stack.pop()] = x
        stack.append(i)
    return res

arr = [17, 18, 5, 4, 6, 1]
print("max bên phải :", Solution().replaceElements(arr[:]))
print("next greater  :", next_greater(arr))

max bên phải : [18, 6, 6, 6, 1, -1]
next greater  : [18, -1, 6, 6, -1, -1]


## 15. Bài tập tự luyện

> Gợi ý ở mục 16, đáp án ở mục 17. Không gọi hàm TODO khi chạy notebook.

### Bài 1 — Tính tay
Kết quả của `replaceElements([2, 9, 3, 9, 1])`? Gán list vào `ex1_answer`.

In [24]:
ex1_answer = None   # TODO

# assert ex1_answer == Solution().replaceElements([2, 9, 3, 9, 1])

### Bài 2 — Max bên trái
Viết `replace_with_left_max(arr)`: thay mỗi phần tử bằng max **bên trái** nó, phần tử đầu thành `-1`. `O(n)`.

In [25]:
def replace_with_left_max(arr: list[int]) -> list[int]:
    # TODO
    pass

# assert replace_with_left_max([17, 18, 5, 4, 6, 1]) == [-1, 17, 18, 18, 18, 18]

### Bài 3 — Đếm "leaders"
Phần tử là **leader** nếu nó **lớn hơn mọi** phần tử bên phải (phần tử cuối luôn là leader). Viết `count_leaders(arr)`.

In [26]:
def count_leaders(arr: list[int]) -> int:
    # TODO: duyệt ngược, so với running max
    pass

# assert count_leaders([17, 18, 5, 4, 6, 1]) == 3      # 18, 6, 1
# assert count_leaders([5, 5, 5]) == 1

### Bài 4 — Best Time to Buy and Sell Stock (LeetCode 121)
Cho `prices`, chọn một ngày mua và một ngày **sau đó** để bán, tối đa lợi nhuận (không có lời trả `0`). Dùng ý tưởng "max bên phải".

In [27]:
def max_profit(prices: list[int]) -> int:
    # TODO
    pass

# assert max_profit([7, 1, 5, 3, 6, 4]) == 5
# assert max_profit([7, 6, 4, 3, 1]) == 0

### Bài 5 — Mini challenge: Trapping Rain Water (LeetCode 42)
Lượng nước đọng ở cột `i` = `min(max_trái_gồm_i, max_phải_gồm_i) - height[i]`. Viết `trap(height)` với hai mảng prefix/suffix max.

In [28]:
def trap(height: list[int]) -> int:
    # TODO
    pass

# assert trap([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]) == 6
# assert trap([4, 2, 0, 3, 2, 5]) == 9

## 16. Gợi ý 3 mức

**Bài 1:** (1) Đi từ phải. (2) `mx` lần lượt: −1, 1, 9, 9, 9. (3) Ghi `mx` **trước** khi cập nhật.

**Bài 2:** (1) Đối xứng với bài chính. (2) Duyệt xuôi. (3) Chính là hàm `wrong_left_to_right` ở mục 13 — nó "sai" cho bài chính nhưng **đúng** cho bài này!

**Bài 3:** (1) Leader ⇔ `arr[i] > max(bên phải)`. (2) Duyệt ngược với `mx = -inf` (hoặc `-1` vì giá trị dương). (3) `if x > mx: cnt += 1; mx = x`.

**Bài 4:** (1) Bán ở ngày có giá cao nhất **sau** ngày mua. (2) Duyệt ngược giữ `best_sell`. (3) `ans = max(ans, best_sell - p)` rồi cập nhật `best_sell`.

**Bài 5:** (1) Hai mảng `left[i] = max(height[:i+1])`, `right[i] = max(height[i:])`. (2) Tính bằng running max hai chiều. (3) Cộng `min(left[i], right[i]) - height[i]`.

## 17. Đáp án tham khảo

In [29]:
ans_ex1 = [9, 9, 9, 1, -1]


def ans_replace_with_left_max(arr):
    res, mx = [], -1
    for x in arr:
        res.append(mx)
        mx = max(mx, x)
    return res


def ans_count_leaders(arr):
    cnt, mx = 0, float("-inf")
    for x in reversed(arr):
        if x > mx:
            cnt += 1
            mx = x
    return cnt


def ans_max_profit(prices):
    best_sell, ans = 0, 0
    for p in reversed(prices):
        ans = max(ans, best_sell - p)
        best_sell = max(best_sell, p)
    return ans


def ans_trap(height):
    n = len(height)
    left, right = [0] * n, [0] * n
    mx = 0
    for i in range(n):
        mx = max(mx, height[i]); left[i] = mx
    mx = 0
    for i in range(n - 1, -1, -1):
        mx = max(mx, height[i]); right[i] = mx
    return sum(min(left[i], right[i]) - height[i] for i in range(n))


assert ans_ex1 == Solution().replaceElements([2, 9, 3, 9, 1])
assert ans_replace_with_left_max([17, 18, 5, 4, 6, 1]) == [-1, 17, 18, 18, 18, 18]
assert ans_count_leaders([17, 18, 5, 4, 6, 1]) == 3
assert ans_count_leaders([5, 5, 5]) == 1
assert ans_max_profit([7, 1, 5, 3, 6, 4]) == 5
assert ans_max_profit([7, 6, 4, 3, 1]) == 0
assert ans_trap([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]) == 6
assert ans_trap([4, 2, 0, 3, 2, 5]) == 9
# đối chiếu brute force
random.seed(42)
for _ in range(500):
    a = [random.randint(1, 10) for _ in range(random.randint(1, 15))]
    assert ans_count_leaders(a) == sum(1 for i in range(len(a)) if all(a[i] > y for y in a[i + 1:]))
    assert ans_max_profit(a) == max([0] + [a[j] - a[i] for i in range(len(a)) for j in range(i + 1, len(a))])
print("Đáp án bài tập: tất cả assert PASS")

Đáp án bài tập: tất cả assert PASS


`float("-inf")` là **âm vô cực** — nhỏ hơn mọi số, giá trị khởi tạo an toàn cho running max khi không biết giới hạn dưới của dữ liệu.

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def replaceElements(self, arr):
        mx = -1
        for i in range(len(arr) - 1, -1, -1):
            arr[i], mx = mx, max(mx, arr[i])
        return arr
```

**Câu thần chú:** *"Đi từ phải, ghi max cũ, rồi mới gộp chính mình."*

**Checklist:**
- [ ] Duyệt **ngược** đủ tới index `0` (`range(n-1, -1, -1)`).
- [ ] `mx` khởi tạo `-1` (đúng yêu cầu phần tử cuối).
- [ ] Ghi `arr[i] = mx` dùng max **cũ** (không gồm `arr[i]`).
- [ ] Không làm mất `arr[i]` gốc trước khi cập nhật `mx` (lưu tạm hoặc gán đồng thời).
- [ ] Test `n = 1`, mảng tăng dần, giảm dần, toàn bằng nhau.